# automatically quantify images
a notebook to automatically quantify things in images

for now we will start with simple tif images and we won't need many or very advanced packages

In [ ]:
pip install scikit-image

In [ ]:
pip install matplotlib

In [ ]:
pip install pandas

Here we will place all of our imports

In [ ]:
from matplotlib import pyplot as plt # for plotting
from skimage.io import imread, imsave # for reading/writing images
import numpy as np # for numerical operations
import os # for file handling
import tifffile # for reading TIFF metadata
from skimage.measure import label, regionprops # for quantifying objects in binary images
import pandas as pd # for data handling

The following code we will use to have access to files from our repository...

In [ ]:

# Clone the repo if not already in Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/NBCimageAnalysis'):
        !git clone https://github.com/FilLieb/NBCimageAnalysis.git
    os.chdir('/content/NBCimageAnalysis/learning/')
    print(os.listdir('.'))

next let's load an image with its associated binary images...

In [ ]:
binary = imread('../data/artificial/test_3col.tif')
image = imread('../data/artificial/test_3col_intensities.tif')

print(binary.shape)
print(image.shape)


we can also check whether there is metadata associated and whether we can find the actual size of a pixel

In [ ]:
with tifffile.TiffFile('../data/artificial/test_3col.tif') as tif:
    page = tif.pages[0]

    # Check for XResolution / YResolution tags
    x_res = page.tags.get('XResolution')
    y_res = page.tags.get('YResolution')

    if x_res:
        # Tags store resolution as a fraction (pixels per unit)
        num, denom = x_res.value
        x_voxel = denom / num  # unit size per pixel
        print(f"X voxel size: {x_voxel}")
    
    if y_res:
        # Tags store resolution as a fraction (pixels per unit)
        num, denom = y_res.value
        y_voxel = denom / num  # unit size per pixel
        print(f"Y voxel size: {y_voxel}")

and let's see what we have

In [ ]:
# make a quick figure to display individual channels
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

axes[0, 0].imshow(image[:,:,0], cmap='gray')
axes[0, 0].set_title("Channel 0")

axes[0, 1].imshow(image[:,:,1], cmap='gray')
axes[0, 1].set_title("Channel 1")

axes[0, 2].imshow(image[:,:,2], cmap='gray')
axes[0, 2].set_title("Channel 2")

axes[1, 0].imshow(binary[:,:,0], cmap='gray')
axes[1, 1].imshow(binary[:,:,1], cmap='gray')
axes[1, 2].imshow(binary[:,:,2], cmap='gray')


plt.tight_layout()
plt.show()

In order to quantify "things" from our images we will use scikit-image. The first thing we need to do is label our image, for example all pixels of the same value belong to one object. This way we can treat them as different entities or clusters.

In [ ]:
labeled = label(binary[:,:,0])
plt.imshow(labeled)

using this labeled image, we can get the "properties" of the individual entities

In [ ]:
regions = regionprops(labeled) #access properties of each labeled region

print(f"The area of the first region is: {regions[0].area}") # print area of first region
print(f"The number of pixels in the first region is: {regions[0].num_pixels}") # print number of pixels in first region
print(f"The maximum feret diameter of the first region is: {regions[0].feret_diameter_max}") # print max feret diameter of first region




If we wanted to convert the area into scaled units, we need to use the info from the metadata of the image.

We have already done the important part of the work and can use the variables x_voxel and y_voxel.

In [ ]:
print(f"The area of the first region is: {regions[0].area*x_voxel*y_voxel} µm²") # print area of first region

We can use the information in regions to calculate the average area or average maximum feret diameter.

In [ ]:
print(f"The average area of all regions is: {np.mean([r.area for r in regions])*x_voxel*y_voxel} µm²") # print average area of all regions
print(f"The average maximum feret diameter of all regions is: {np.mean([r.feret_diameter_max for r in regions])*x_voxel} µm") # print average feret diameter of all regions


If we simply want to know the number of clusters, we can check the length (i.e. number of objects)...

In [ ]:
print(f"There are {len(regions)} clusters in this image.") # print total number of regions

We can also use our labeled image and measure the intensities in the corresponding original image...

In [ ]:
props = regionprops(labeled, intensity_image=image[:,:,0])#access properties of each labeled region, using intensity image for intensity-based measurements
print(f" The mean intensity of the first region is: {props[0].intensity_mean}") # print mean intensity of first region
print(f" The maximum intensity of the first region is: {props[0].intensity_max}") # print max intensity of first region
print(f" The minimum intensity of the first region is: {props[0].intensity_min}") # print min intensity of first region

We could also ask the question of how many of our three clusters are actually synaptic cluster, i.e. "touching" a presynaptic partner. 

In [ ]:
gphn = binary[:,:,0]
gphn_labeled = label(gphn)

vgat = binary[:,:,1]

# Find where gphn and vgat overlap
overlap = (gphn_labeled > 0) & (vgat > 0) # find pixels where both gphn and vgat are positive

plt.imshow(overlap, cmap='gray')

# Find which gphn labels are present in the overlap region
overlapping_labels = np.unique(gphn_labeled[overlap]) # get unique labels of gphn that overlap with vgat
overlapping_labels = overlapping_labels[overlapping_labels != 0]  # remove background

print(f"Found {len(overlapping_labels)} out of {len(regions)} synaptic clusters. Which is {len(overlapping_labels)/len(regions)*100:.2f}% of all clusters.")

We can now create a labeled binary image of the gephyrin clusters that have a presynaptic partner.

In [ ]:
# Build a mask containing only the gphn clusters that overlap with vgat
gphn_vgat_positive = np.isin(gphn_labeled, overlapping_labels)

# Re-label this filtered mask
gphn_vgat_labeled = label(gphn_vgat_positive)

plt.imshow(gphn_vgat_labeled)

We can now use the same code as above if we are only interested in the intensity of the synaptic clusters and simply use our newly generated labeled mask:

In [ ]:
props_synaptic = regionprops(gphn_vgat_labeled, intensity_image=image[:,:,0]) #note that we have used here the gphn_vgat_labeled image to get properties only of the overlapping clusters, and we have used the original image as intensity image to get intensity-based measurements for these clusters
print(f" The mean intensity of the first synaptic region is: {props_synaptic[0].intensity_mean}") # print mean intensity of first region
print(f" The maximum intensity of the first synaptic region is: {props_synaptic[0].intensity_max}") # print max intensity of first region
print(f" The minimum intensity of the first synaptic region is: {props_synaptic[0].intensity_min}") # print min intensity of first region

We should also measure the size of the cell:

In [ ]:
cell = binary[:,:,2]
cell_labeled = label(cell)

cell_regions = regionprops(cell_labeled) #access properties of each labeled region

print(f"The total area of the cell(s) is: {np.mean([r.area for r in cell_regions])*x_voxel*y_voxel} µm²")

Let's summarize our analysis:

In [ ]:
number_of_gphn_clusters = len(regions)
number_of_gphn_vgat_clusters = len(props_synaptic)

average_area_gphn_clusters = np.mean([r.area for r in regions])*x_voxel*y_voxel
average_area_gphn_vgat_clusters = np.mean([r.area for r in props_synaptic])*x_voxel*y_voxel

average_intensity_gphn_clusters = np.mean([r.intensity_mean for r in props]) # average intensity of all gphn clusters
average_intensity_gphn_vgat_clusters = np.mean([r.intensity_mean for r in props_synaptic]) # average intensity of all synaptic clusters

total_cell_area = np.mean([r.area for r in cell_regions])*x_voxel*y_voxel

# Create a DataFrame to store the results
results_df = pd.DataFrame({
    '# gphn clusters': [number_of_gphn_clusters],
    '# synaptic gphn clusters': [number_of_gphn_vgat_clusters],
    'area gphn clusters [µm²]': [average_area_gphn_clusters],
    'area synaptic gphn clusters [µm²]': [average_area_gphn_vgat_clusters],
    'intensity gphn clusters': [average_intensity_gphn_clusters],
    'intensity synaptic gphn clusters': [average_intensity_gphn_vgat_clusters],
    'total cell area [µm²]': [total_cell_area]
})

print(results_df)


In [ ]:
summary_view = (
    results_df.T
    .rename(columns={0: "value"})
    .round(3)
)
display(summary_view)